In [ ]:
import sys
import os
import math

from matplotlib import pyplot as plt

import numpy as np

from sklearn.decomposition import PCA
from sklearn.cross_decomposition import CCA


sys.path.append(os.path.join(os.getcwd(), '../import'))

from utils import read_utils
from utils import train_utils

In [ ]:
path = os.path.join(read_utils.get_data_path(), 'worm-2025-fieseler', 'exported_data_')
filenames, continuous_behaviors, discrete_states, eigenworms, kymographs, traces = read_utils.get_data(path)

In [ ]:
selection = ['REV', 'VENTRAL_TURN', 'DORSAL_TURN', 'PAUSE']

modes_manifold = list()
traces_manifold = list()
vars_expl_rs = list()  # variance explained with respect to the reconstructed PCA data
vars_expl = list()
for trace, ds in zip(traces, discrete_states):

	pca = PCA(n_components=5)
	trace_reduced_pca = pca.fit_transform(trace)
	trace_reconstructed_pca = pca.inverse_transform(trace_reduced_pca)

	cca = CCA(n_components=2, scale=True)
	trace_reduced, ds_reduced = cca.fit_transform(trace_reconstructed_pca, ds[selection].values)
	trace_reconstructed, ds_reconstructed = cca.inverse_transform(trace_reduced, ds_reduced)
	modes_manifold.append((trace_reduced, ds_reduced))
	traces_manifold.append((trace_reconstructed, ds_reconstructed))

	var_ratio_rs = train_utils.get_variance_explained(trace_reconstructed_pca, trace_reconstructed)
	var_ratio = train_utils.get_variance_explained(trace, trace_reconstructed)
	vars_expl_rs.append(var_ratio_rs)
	vars_expl.append(var_ratio)


n_plots = 36
dim = math.ceil(math.sqrt(n_plots))
fig, axes = plt.subplots(dim, dim, figsize=(20, 24))
for i in range(n_plots):
	axes[i // dim, i % dim].set_xticklabels([])
	axes[i // dim, i % dim].set_yticklabels([])
	axes[i // dim, i % dim].grid(False)

scale = [-1, 0, 1]
for i, (X_c, Y_c) in enumerate(modes_manifold):
	axes[i // dim, i % dim].plot(X_c[:, 0], X_c[:, 1])
	axes[i // dim, i % dim].plot(Y_c[:, 0], Y_c[:, 1])
	axes[i // dim, i % dim].set_xticks(scale)
	axes[i // dim, i % dim].set_xticklabels(scale)
	axes[i // dim, i % dim].set_yticks(scale)
	axes[i // dim, i % dim].set_yticklabels(scale)
	axes[i // dim, i % dim].grid(True)
	axes[i // dim, i % dim].set_title(f'{filenames[i][:-14]}\n{i} ({vars_expl[i]:.2f}; {vars_expl_rs[i]:.2f})', fontsize=8)

plt.subplots_adjust(hspace=0.5)
print(np.mean(np.array(vars_expl)), np.mean(np.array(vars_expl_rs)))

In [ ]:
selection = ['REV', 'VENTRAL_TURN', 'DORSAL_TURN', 'PAUSE']

modes_manifold_cca = list()
traces_manifold_cca = list()
vars_expl_cca = list()
for trace, ds in zip(traces, discrete_states):

	cca = CCA(n_components=2, scale=True)
	trace_reduced_cca, ds_reduced_cca = cca.fit_transform(trace, ds[selection].values)
	trace_reconstructed_cca, ds_reconstructed_cca = cca.inverse_transform(trace_reduced_cca, ds_reduced_cca)
	modes_manifold_cca.append((trace_reduced_cca, ds_reduced_cca))
	traces_manifold_cca.append((trace_reconstructed_cca, ds_reconstructed_cca))

	var_ratio = train_utils.get_variance_explained(trace, trace_reconstructed_cca)
	vars_expl_cca.append(var_ratio)


n_plots = 36
dim = math.ceil(math.sqrt(n_plots))
fig, axes = plt.subplots(dim, dim, figsize=(20, 24))
for i in range(n_plots):
	axes[i // dim, i % dim].set_xticklabels([])
	axes[i // dim, i % dim].set_yticklabels([])
	axes[i // dim, i % dim].grid(False)

scale = [-1, 0, 1]
for i, (X_c, Y_c) in enumerate(modes_manifold_cca):
	axes[i // dim, i % dim].plot(X_c[:, 0], X_c[:, 1])
	#axes[i // dim, i % dim].plot(Y_c[:, 0], Y_c[:, 1])
	axes[i // dim, i % dim].set_xticks(scale)
	axes[i // dim, i % dim].set_xticklabels(scale)
	axes[i // dim, i % dim].set_yticks(scale)
	axes[i // dim, i % dim].set_yticklabels(scale)
	axes[i // dim, i % dim].grid(True)
	axes[i // dim, i % dim].set_title(f'{filenames[i][:-14]}\n{i} ({vars_expl_cca[i]:.2f})', fontsize=8)

plt.subplots_adjust(hspace=0.5)
print(np.mean(np.array(vars_expl_cca)))